# Episode 2 — Vectorized `jit`, Timing Matmul, and jaxpr

**Student workbook** · code along with the video.

Episode 1 built an eager forward pass. Here we **compile** with `jax.jit`, **batch** with `jax.vmap`, inspect **jaxprs**, and **time** eager vs compiled matmul.

| | |
|---|---|
| **Prereq** | Episode 1 — PRNG keys, `params`, broadcasting |
| **Next** | Episode 3 — `scan` vs Python loop (prefix sum) |

**JAX docs:** [Tracing](https://docs.jax.dev/en/latest/tracing.html) · [Key concepts — jaxpr & XLA](https://docs.jax.dev/en/latest/key-concepts.html) · [`jax.jit`](https://docs.jax.dev/en/latest/_autosummary/jax.jit.html) · [`jax.vmap`](https://docs.jax.dev/en/latest/_autosummary/jax.vmap.html) · [`jax.make_jaxpr`](https://docs.jax.dev/en/latest/_autosummary/jax.make_jaxpr.html)


In [ ]:
# your code here


## What `jax.jit` is doing

**`jax.jit` does not "run Python faster."** On the first call with given array **shapes and dtypes**, JAX **traces** your function: array values are replaced by **tracers**, and each `jnp` op records a primitive into a **jaxpr** (JAX program). That jaxpr is **lowered** to **StableHLO**, then **XLA** (Accelerated Linear Algebra) optimizes and emits code for your device (CPU/GPU/TPU). Later calls with the **same shapes/dtypes** reuse the compiled executable; new shapes trigger **recompilation**.

Read: [Tracing](https://docs.jax.dev/en/latest/tracing.html) · [Key concepts — jaxpr & XLA](https://docs.jax.dev/en/latest/key-concepts.html) · [JAX internals: jaxpr](https://docs.jax.dev/en/latest/jaxpr.html)


## JIT limitations (preview)

Keep these in mind whenever you reach for `jit`:

1. **Only traced ops compile** — arbitrary Python (`for`, `print`, file I/O) does not magically speed up; only `jnp` / `lax` primitives recorded during tracing become XLA code.
2. **One executable per shape/dtype** — change `B` or matrix sizes and JAX **re-traces** (compile again).
3. **Pure functions** — no in-place array writes (`x[0] = 1` fails); use `.at[...].set(...)` and return the new array (Episode 0).
4. **Timing is tricky** — JAX dispatches asynchronously; call `jax.block_until_ready(...)` before stopping the clock on GPU/TPU.


In [ ]:
# your code here


## Inspect jaxprs

Compare the primitives JAX records for eager trace vs a `jit`-wrapped function.


In [ ]:
# your code here


## Time eager vs `jit`

Warm up once, then time steady-state. The **first** call on fresh `jit(fn)` includes trace + compile.


In [ ]:
# your code here


## Recompilation on new shapes

Call the same jitted function with different `(M, K, N)` — JAX builds a new executable.


In [ ]:
# your code here


## Batched matmul with `vmap`

Vectorize over a leading batch axis: `(B, M, K) @ (B, K, N) → (B, M, N)`.


In [ ]:
# your code here


## `jit` ∘ `vmap`

Compile the whole batched kernel in one shot.


In [ ]:
# your code here


## Batched forward (Episode 1)

`vmap` over the batch dimension of `x`; `params` is shared (`in_axes=(None, 0)`).


In [ ]:
# your code here


---
## Exercise

1. `make_jaxpr` on `jax.jit(jax.vmap(forward, in_axes=(None, 0)))` for `B=64` — find a `dot_general` (or similar) line in the output. Paste one jaxpr line in a comment.
2. `timeit` eager `vmap(forward)` vs `jit(vmap(forward))` after warmup (warm up once; separate first call from steady-state average).


In [ ]:
# your code here


**Next:** Episode 3 — prefix sum with a Python loop vs `lax.scan` (no `jit`).
